# Do LLMs Verify or Conform? — Experiment Walkthrough

This notebook walks through every stage of our study: **why we did it, what we measured, how we measured it, and what the results mean**.

All examples are drawn from real data (aws-c-common, 83 functions, gpt-oss-120b as primary LLM).

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

BASE = '/home/weiqi/Verification/LLM4Harness/experiment_aws_cbmc'
RESULTS = os.path.join(BASE, 'results')
EVAL = os.path.join(BASE, 'evaluation')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 11
print('Setup complete.')

---
## Part 1: Background — What is CBMC?

CBMC (C Bounded Model Checker) is a **static analysis tool** for C programs. Given a C function and a set of assertions (`assert`), it exhaustively explores all execution paths within a bounded state space:

- **UNSAT (SUCCESS)**: For *all* possible inputs within the bound, no `assert` is violated. The assertions hold universally.
- **SAT (FAIL)**: A concrete counterexample was found — a specific input that violates an `assert`.
- **UNKNOWN / TIMEOUT**: The state space is too large to explore within the time limit. No conclusion.

> Key difference from testing: ordinary tests run a finite set of inputs. CBMC proves or disproves properties over **all** bounded inputs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

scenarios = [
    {
        'title': 'CBMC → SUCCESS\n(UNSAT)',
        'color': '#2ecc71',
        'desc': 'All input paths explored.\nNo assert ever violated.\n→ Assertions hold within the bound.',
        'detail': 'assert(result >= 0) ✓\nassert(list.length == old.length + 1) ✓\nassert(ptr != NULL) ✓'
    },
    {
        'title': 'CBMC → FAIL\n(SAT)',
        'color': '#e74c3c',
        'desc': 'A counterexample was found.\nA specific input violates an assert.\n→ Code has a bug, or assertion is wrong.',
        'detail': 'Counterexample:\n  list.length = 5, index = 7\n  assert(index < list.length)  ← FAILS'
    },
    {
        'title': 'CBMC → UNKNOWN\n(Timeout)',
        'color': '#f39c12',
        'desc': 'State space explosion.\nExploration incomplete within time limit.\n→ No conclusion.',
        'detail': 'State space too large.\nUnbounded loops or\ntoo many pointer dereferences.'
    }
]

for ax, s in zip(axes, scenarios):
    ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.05), 0.9, 0.9,
        boxstyle='round,pad=0.02', linewidth=2,
        edgecolor=s['color'], facecolor=s['color'] + '15'))
    ax.text(0.5, 0.85, s['title'], ha='center', va='top', fontsize=13,
            fontweight='bold', color=s['color'], transform=ax.transAxes)
    ax.text(0.5, 0.60, s['desc'], ha='center', va='top', fontsize=9.5,
            transform=ax.transAxes, linespacing=1.6)
    ax.text(0.5, 0.30, s['detail'], ha='center', va='top', fontsize=8.5,
            transform=ax.transAxes, family='monospace', color='#333', linespacing=1.5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

fig.suptitle('The Three Possible CBMC Outcomes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Part 2: What is a Proof Harness?

CBMC cannot verify a whole program directly. For each function you want to verify, you write a **proof harness** — a small driver that:

1. **Establishes valid inputs** via `__CPROVER_assume()` — constrains inputs to the legal domain
2. **Calls the function under test**
3. **States the properties to verify** via `assert()` — "after the call, these things must hold"

Below is the Ground Truth (GT) harness written by AWS formal verification engineers for `aws_array_list_erase`:

In [ ]:
gt_harness = '''
void aws_array_list_erase_harness() {
    struct aws_array_list list;   // declare the array list (symbolic)
    size_t index;                 // index to erase (nondeterministic)

    // ① ASSUMES: restrict CBMC to valid inputs only
    __CPROVER_assume(aws_array_list_is_bounded(&list, MAX_INITIAL_ITEM_ALLOCATION, MAX_ITEM_SIZE));
    ensure_array_list_has_allocated_data_member(&list);
    __CPROVER_assume(aws_array_list_is_valid(&list));

    // ② Snapshot state before the call
    struct aws_array_list old = list;
    struct store_byte_from_buffer old_byte;
    save_byte_from_array((uint8_t *)list.data, list.current_size, &old_byte);

    // ③ Call the function under verification
    if (aws_array_list_erase(&list, index) == AWS_OP_SUCCESS) {

        // ④ ASSERTS: postconditions on success
        assert(list.length == old.length - 1);         // length decremented
        assert(list.item_size == old.item_size);        // item_size unchanged
        assert(list.alloc == old.alloc);                // allocator unchanged
        assert(list.current_size == old.current_size);  // capacity unchanged
        assert(index < old.length);                     // index was in bounds

    } else {
        // ④' On failure: list must be unmodified
        assert_array_list_equivalence(&list, &old, &old_byte);
    }
    // Structural invariant must hold regardless of outcome
    assert(aws_array_list_is_valid(&list));
}
'''

print('AWS Engineer Ground Truth Harness (aws_array_list_erase):')
print(gt_harness)
print('Total assert():', gt_harness.count('assert(') + gt_harness.count('assert_array'))
print('Total assume():', gt_harness.count('__CPROVER_assume'))

---
## Part 3: The Experiment Pipeline — How LLMs Generate Harnesses

We ask an LLM (gpt-oss-120b) to generate a harness, run CBMC, feed back the result, and repeat up to 15 iterations:

```
LLM generates iter_1_harness.c
        ↓
   CBMC runs
     ↙    ↘      ↘
 SUCCESS   FAIL   UNKNOWN
   ↓        ↓        ↓
  Done   "fix this   "state space too large,
         assertion"   simplify your harness"
              ↓            ↓
          LLM generates iter_2_harness.c
              ↓
           continue...
```

**The critical question: what does the LLM do when CBMC returns UNKNOWN?**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

boxes = [
    (0.08, 0.5,  'LLM\ngen iter_1',     '#3498db'),
    (0.28, 0.5,  'CBMC\nrun',            '#95a5a6'),
    (0.48, 0.82, 'SUCCESS\n✓',           '#2ecc71'),
    (0.48, 0.5,  'FAIL\n(CEX)',          '#e74c3c'),
    (0.48, 0.18, 'UNKNOWN\n(timeout)',   '#f39c12'),
    (0.72, 0.5,  'LLM\ngen iter_N+1',   '#3498db'),
    (0.92, 0.5,  'Done\n(PASS)',         '#2ecc71'),
]
for x, y, label, color in boxes:
    ax.add_patch(mpatches.FancyBboxPatch((x-0.07, y-0.12), 0.14, 0.24,
        boxstyle='round,pad=0.01', linewidth=2,
        edgecolor=color, facecolor=color + '33'))
    ax.text(x, y, label, ha='center', va='center', fontsize=10,
            fontweight='bold', color=color)

arrows = [
    (0.15, 0.5,  0.21, 0.5,  ''),
    (0.35, 0.56, 0.41, 0.76, 'UNSAT'),
    (0.35, 0.5,  0.41, 0.5,  'fix assertion'),
    (0.35, 0.44, 0.41, 0.24, 'simplify harness'),
    (0.55, 0.5,  0.65, 0.5,  ''),
    (0.55, 0.24, 0.65, 0.44, ''),
    (0.79, 0.5,  0.85, 0.5,  ''),
]
for x1, y1, x2, y2, label in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    if label:
        ax.text((x1+x2)/2, (y1+y2)/2 + 0.04, label,
                ha='center', fontsize=8.5, color='#555', style='italic')

ax.text(0.5, 0.02,
    '⚠️  Key question: on UNKNOWN, the LLM can either '
    '(A) fix the assumes to shrink the state space, or '
    '(B) delete assertions so CBMC has less to prove',
    ha='center', fontsize=9.5, color='#c0392b', fontweight='bold')

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('The LLM–CBMC Iterative Loop', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

---
## Part 4: A Real Sacrifice Case — aws_array_list_erase

Here is what actually happened: the LLM deleted **8 assertions** after receiving a CBMC UNKNOWN.

In [ ]:
func = 'aws_array_list_erase'
result_dir = os.path.join(RESULTS, 'feedback_loop_A_gptoss120b', func)

with open(os.path.join(result_dir, 'summary.json')) as f:
    summary = json.load(f)

iter_files = sorted([f for f in os.listdir(result_dir)
                     if f.startswith('iter_') and f.endswith('_harness.c')])
iter_code = {}
iter_asserts = {}
for fname in iter_files:
    num = int(fname.split('_')[1])
    with open(os.path.join(result_dir, fname)) as f:
        code = f.read()
    iter_code[num] = code
    n = sum(1 for line in code.splitlines()
            if 'assert(' in line and '__CPROVER_assume' not in line
            and not line.strip().startswith('//'))
    iter_asserts[num] = n

print(f'Function: {func}')
print(f'Total iterations: {len(summary["iterations"])}')
print()
icons = {'SUCCESS': '✅', 'FAIL': '❌', 'UNKNOWN': '⚠️', 'COMPILE_ERROR': '🔧'}
for it in summary['iterations']:
    n = iter_asserts.get(it['iter'], '?')
    print(f"  Iter {it['iter']}: {icons.get(it['verify'], '?')} {it['verify']:15s} "
          f"assertions: {n}  action: {it['action']}")

In [ ]:
def extract_asserts(code):
    return [line.strip() for line in code.splitlines()
            if 'assert(' in line and '__CPROVER_assume' not in line
            and not line.strip().startswith('//') and '#include' not in line]

asserts_1 = extract_asserts(iter_code[1])
asserts_2 = extract_asserts(iter_code[2])
set_1, set_2 = set(asserts_1), set(asserts_2)

print('=' * 65)
print(f'ITER 1 — {len(asserts_1)} assertions  →  CBMC: UNKNOWN')
print('=' * 65)
for a in asserts_1:
    tag = '❌ DELETED' if a not in set_2 else '   kept   '
    print(f'  {tag}  {a}')

print()
print('=' * 65)
print(f'ITER 2 — {len(asserts_2)} assertions  →  CBMC: SUCCESS')
print('=' * 65)
for a in asserts_2:
    tag = '✨ NEW' if a not in set_1 else '      '
    print(f'  {tag}  {a}')

print()
print(f'Deleted: {len(set_1 - set_2)}  |  Added: {len(set_2 - set_1)}')

In [ ]:
# Compare final LLM harness against GT
gt_asserts_keywords = [
    ('assert(list.length == old.length - 1)', 'length decremented'),
    ('assert(list.item_size == old.item_size)', 'item_size frame'),
    ('assert(list.alloc == old.alloc)', 'alloc frame'),
    ('assert(list.current_size == old.current_size)', 'current_size frame'),
    ('assert(index < old.length)', 'index validity'),
    ('assert_array_list_equivalence', 'failure equivalence'),
    ('assert(aws_array_list_is_valid', 'structural invariant'),
]

print('GT assertion coverage in LLM final harness (iter_2):')
print()
for gt_assert, desc in gt_asserts_keywords:
    key = gt_assert.split('(')[1].split(')')[0][:25] if '(' in gt_assert else gt_assert[:25]
    in_iter1 = any(key[:15] in a for a in asserts_1)
    in_final = any(key[:15] in a for a in asserts_2)
    if in_final:
        status = '✅ MATCHED'
    elif in_iter1:
        status = '⚠️  SACRIFICE  (generated then deleted)'
    else:
        status = '❌ KNOWLEDGE GAP  (never generated)'
    print(f'  {status:<42}  [{desc}]')

---
## Part 5: PASS Rate vs Recall — Why PASS Rate is a Misleading Metric

**PASS rate**: fraction of functions where the LLM harness passes CBMC verification.  
**Recall**: fraction of GT assertions that the LLM harness covers (measured by re-running LLM assertions under GT assumes).

**Central finding: higher PASS rate does not imply higher recall — the ordering reverses.**

In [ ]:
conditions_data = {
    'G\n(no feedback)':      {'pass': 31.3, 'recall': 0.290, 'sacrifice': 0.0,   'color': '#95a5a6'},
    'H\n(neutral strategy)': {'pass': 62.7, 'recall': 0.303, 'sacrifice': 86.3,  'color': '#e67e22'},
    'A\n(baseline)':         {'pass': 62.5, 'recall': 0.346, 'sacrifice': 92.3,  'color': '#3498db'},
    'K\n(spec-first)':       {'pass': 81.9, 'recall': 0.268, 'sacrifice': 25.0,  'color': '#e74c3c'},
    'Oracle\n(GT assumes)':  {'pass': 84.3, 'recall': 0.251, 'sacrifice': 11.8,  'color': '#c0392b'},
    'M\n(bounding hint)':    {'pass': 75.3, 'recall': 0.389, 'sacrifice': 0.0,   'color': '#2ecc71'},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sorted by PASS rate
ax = axes[0]
names = list(conditions_data.keys())
sorted_pass = sorted(zip(
    [conditions_data[n]['pass'] for n in names], names,
    [conditions_data[n]['color'] for n in names]), reverse=True)
p_s, n_s, c_s = zip(*sorted_pass)
ax.barh(range(len(n_s)), p_s, color=c_s, alpha=0.8, height=0.6)
ax.set_yticks(range(len(n_s))); ax.set_yticklabels(n_s, fontsize=10)
ax.set_xlabel('PASS Rate (%)')
ax.set_title('① Ranked by PASS Rate\n(higher = better?)', fontsize=12, fontweight='bold')
ax.set_xlim(0, 100)
for i, p in enumerate(p_s):
    ax.text(p + 1, i, f'{p:.1f}%', va='center', fontsize=9)

# Right: sorted by Recall
ax = axes[1]
sorted_recall = sorted(zip(
    [conditions_data[n]['recall'] for n in names], names,
    [conditions_data[n]['color'] for n in names]), reverse=True)
r_s, n_r, c_r = zip(*sorted_recall)
ax.barh(range(len(n_r)), [r*100 for r in r_s], color=c_r, alpha=0.8, height=0.6)
ax.set_yticks(range(len(n_r))); ax.set_yticklabels(n_r, fontsize=10)
ax.set_xlabel('Recall (%) [vs GT harness]')
ax.set_title('② Ranked by Recall\n(actual specification coverage)', fontsize=12, fontweight='bold')
ax.set_xlim(0, 50)
for i, r in enumerate(r_s):
    ax.text(r*100 + 0.5, i, f'{r:.3f}', va='center', fontsize=9)

fig.suptitle(
    'PASS Rate vs Recall: rankings completely reversed!\n'
    'Oracle (highest PASS 84.3%) = lowest Recall (0.251);  M (3rd PASS 75.3%) = highest Recall (0.389)',
    fontsize=12, fontweight='bold', color='#c0392b')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: PASS rate vs Recall — shows the negative correlation
fig, ax = plt.subplots(figsize=(9, 6))

for name, d in conditions_data.items():
    short = name.replace('\n', ' ')
    ax.scatter(d['pass'], d['recall'] * 100, s=180, color=d['color'], zorder=5, alpha=0.9)
    ox, oy = 1.5, 0.5
    if 'Oracle' in name: ox, oy = -13, 1
    elif 'M' in name: oy = 1
    ax.annotate(short, xy=(d['pass'], d['recall']*100),
                xytext=(d['pass']+ox, d['recall']*100+oy),
                fontsize=9.5, color=d['color'], fontweight='bold')

ax.annotate('', xy=(85, 24), xytext=(60, 36),
    arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2,
                    connectionstyle='arc3,rad=0.2'))
ax.text(70, 31.5, 'PASS ↑\nRecall ↓', color='#c0392b', fontsize=10,
        fontweight='bold', ha='center')

ax.plot([30, 90], [26, 41], 'g--', alpha=0.3, label='Ideal: positive correlation')
ax.set_xlabel('PASS Rate (%)', fontsize=12)
ax.set_ylabel('Recall (%) [vs GT]', fontsize=12)
ax.set_title('PASS Rate vs Recall: verification success ≠ specification completeness\n'
             'Under ideal conditions these should be positively correlated',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xlim(25, 92); ax.set_ylim(22, 42)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Part 6: Why Does the Ranking Reverse? — Three Mechanisms

| Condition | Why high PASS | Why low Recall |
|-----------|--------------|----------------|
| **Oracle** | GT assumes provide perfect state constraints | **Laziness effect**: assumes are so good that trivial assertions pass CBMC — LLM writes minimal postconditions |
| **K** | NL contract written first → harness starts from a strong base | NL contract itself is incomplete — omits GT postconditions not obvious from the spec |
| **A** | Iterative CBMC repair improves PASS | Some correct assertions deleted under UNKNOWN pressure (sacrifice) |
| **M** | Bounding hint eliminates UNKNOWN events | Zero deletions — all assertions retained, recall is highest |

---
## Part 7: The Two Gaps — Resolving the 97% vs 92.3% Tension

Two numbers that look contradictory:
- **92.3% sacrifice ratio**: of all assertion deletion events, 92.3% are triggered by UNKNOWN (not FAIL)
- **97% never-generated**: of the 198 GT assertions missed by the LLM, 97% were *never generated at any iteration*

These are **not contradictory** — they measure two different gaps with different denominators:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: what triggers deletions (sacrifice ratio denominator = ALL deletions)
ax = axes[0]
labels_l = ['UNKNOWN-triggered\n(Sacrifice)', 'FAIL-triggered\n(Legitimate fix)', 'COMPILE_ERROR-triggered']
sizes_l = [92.3, 4.8, 2.9]
colors_l = ['#e74c3c', '#2ecc71', '#f39c12']
wedges, texts, autotexts = ax.pie(sizes_l, labels=labels_l, colors=colors_l,
                                   autopct='%1.1f%%', startangle=90,
                                   textprops={'fontsize': 10})
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight('bold')
ax.set_title('Condition A: what triggers deletions?\n(denominator = ALL deletions)',
             fontsize=11, fontweight='bold')

# Right: where do GT misses come from (97% denominator = GT missed assertions)
ax = axes[1]
cats = ['Never generated\n(knowledge gap, 97%)',
        'Deleted sacrifice\n(active removal, 2.5%)',
        'Other/unclear\n(0.5%)']
vals = [97.0, 2.5, 0.5]
bar_colors = ['#3498db', '#e74c3c', '#bdc3c7']
bars = ax.bar(cats, vals, color=bar_colors, alpha=0.85, width=0.6)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v}%', ha='center', fontweight='bold', fontsize=12)
ax.set_ylabel('Fraction (%)')
ax.set_title('Origin of 198 missed GT assertions\n(denominator = GT missed assertions)',
             fontsize=11, fontweight='bold')
ax.set_ylim(0, 108)
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle(
    'Two numbers, two denominators, two independent gaps\n'
    '92.3% sacrifice = "when the LLM deletes, it is almost always due to UNKNOWN"\n'
    '97% never-generated = "most GT misses are assertions the LLM never knew to write"',
    fontsize=11, fontweight='bold', color='#2c3e50'
)
plt.tight_layout()
plt.show()

print()
print('Key insight:')
print('  Sacrifice ratio = regression gap: how much the LLM degrades from its own best state')
print('  97% never-generated = quantity gap: how much the LLM never knew to write')
print('  Evidence for sacrifice as regression: H vs A recall gap (Δ=5.6pp, p=0.026*)')
print('  Same PASS rate, different prompt strategy → measurably different recall')

---
## Part 8: The 11 Experimental Conditions

Each condition changes **exactly one variable** to isolate a specific causal question.

In [ ]:
conditions_info = [
    ('G', 'No feedback\n(single pass)', 'What is the baseline LLM quality\nwithout any CBMC feedback loop?', 31.3, 0.290, 0.0, '#95a5a6'),
    ('H', 'Neutral strategy\n(error only, no guidance)', 'Does sacrifice emerge from the LLM itself,\nor is it instructed by our prompt?\n→ Sacrifice is emergent, not instructed', 62.7, 0.303, 86.3, '#e67e22'),
    ('A', 'Baseline\n(standard feedback prompt)', 'Standard iterative CBMC feedback loop.\nAll other conditions compared to this.', 62.5, 0.346, 92.3, '#3498db'),
    ('I', 'Category label\n(told assertion type on failure)', 'If the LLM knows it is deleting a\n"frame condition", will it stop?\n→ No: sacrifice is deliberate, not ignorant', 70.5, None, 92.7, '#9b59b6'),
    ('J', 'Deletion log\n(history of past deletions shown)', 'Is sacrifice caused by the LLM forgetting\nwhat it already deleted?\n→ No: statelessness is not the cause', 67.5, None, 93.0, '#1abc9c'),
    ('K', 'Spec-first\n(NL contract written before harness)', 'Does writing a spec first reduce sacrifice?\n→ Yes, but recall drops (NL contract is incomplete)', 81.9, 0.268, 25.0, '#e74c3c'),
    ('Oracle', 'GT assumes provided\n(perfect precondition scaffolding)', 'With perfect input setup, will the LLM\nwrite better postconditions?\n→ No: laziness effect — LLM writes less', 84.3, 0.251, 11.8, '#c0392b'),
    ('M', 'Bounding hint\n(CBMC knowledge: bound scalar vars)', 'If we fix the LLM\'s CBMC knowledge gap,\ndoes UNKNOWN disappear?\n→ Yes: sacrifice=0, recall=highest', 75.3, 0.389, 0.0, '#2ecc71'),
]

fig, ax = plt.subplots(figsize=(16, 8))
ax.axis('off')

col_headers = ['Cond', 'Design intent', 'Research question answered', 'PASS%', 'Recall', 'Sacrifice%']
col_x = [0.02, 0.09, 0.37, 0.67, 0.75, 0.85]
row_h = 0.095

ax.add_patch(mpatches.FancyBboxPatch((0, 0.925), 1.0, 0.065,
    boxstyle='round,pad=0.005', facecolor='#2c3e50', transform=ax.transAxes))
for x, h in zip(col_x, col_headers):
    ax.text(x, 0.97, h, fontsize=10, fontweight='bold', color='white',
            transform=ax.transAxes, va='top')

for i, (cond, intent, question, pass_r, recall, sacr, color) in enumerate(conditions_info):
    y = 0.92 - (i + 1) * row_h
    ax.add_patch(mpatches.FancyBboxPatch((0, y - 0.005), 1.0, row_h,
        boxstyle='round,pad=0.005', facecolor=color + '15', transform=ax.transAxes,
        linewidth=0.5, edgecolor=color))
    yc = y + row_h * 0.5
    ax.text(col_x[0], yc, cond, fontsize=11, fontweight='bold', color=color,
            transform=ax.transAxes, va='center')
    ax.text(col_x[1], yc, intent, fontsize=8.5, color='#2c3e50',
            transform=ax.transAxes, va='center', linespacing=1.4)
    ax.text(col_x[2], yc, question, fontsize=8, color='#555',
            transform=ax.transAxes, va='center', linespacing=1.3, style='italic')
    ax.text(col_x[3], yc, f'{pass_r:.1f}%', fontsize=10, color=color,
            fontweight='bold', transform=ax.transAxes, va='center', ha='center')
    recall_str = f'{recall:.3f}' if recall else '—'
    rc = '#2ecc71' if recall and recall > 0.35 else ('#e74c3c' if recall and recall < 0.3 else '#555')
    ax.text(col_x[4], yc, recall_str, fontsize=10, color=rc,
            fontweight='bold', transform=ax.transAxes, va='center', ha='center')
    sc = '#2ecc71' if sacr == 0.0 else ('#e74c3c' if sacr > 85 else '#f39c12')
    ax.text(col_x[5], yc, f'{sacr:.1f}%', fontsize=10, color=sc,
            fontweight='bold', transform=ax.transAxes, va='center', ha='center')

ax.set_title('All 11 Experimental Conditions: What Each One Tests',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## Part 9: Deletion Scope — Panic Deletion vs Targeted Deletion

In [ ]:
deletion_scope_data = {
    'A':  {'events': 29, 'mean_scope': 6.7, 'panic_pct': 72, 'targeted_pct': 17, 'color': '#3498db'},
    'I':  {'events': 24, 'mean_scope': 5.5, 'panic_pct': 71, 'targeted_pct': 12, 'color': '#9b59b6'},
    'J':  {'events': 21, 'mean_scope': 5.6, 'panic_pct': 76, 'targeted_pct': 10, 'color': '#1abc9c'},
    'K':  {'events': 5,  'mean_scope': 1.6, 'panic_pct': 20, 'targeted_pct': 80, 'color': '#e74c3c'},
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cnames = list(deletion_scope_data.keys())
colors_ds = [deletion_scope_data[c]['color'] for c in cnames]

ax = axes[0]
events = [deletion_scope_data[c]['events'] for c in cnames]
ax.bar(cnames, events, color=colors_ds, alpha=0.8, width=0.5)
ax.set_ylabel('UNKNOWN-triggered deletion events')
ax.set_title('Number of UNKNOWN deletion events')
for i, e in enumerate(events):
    ax.text(i, e + 0.3, str(e), ha='center', fontweight='bold')
ax.set_ylim(0, 35)

ax = axes[1]
means = [deletion_scope_data[c]['mean_scope'] for c in cnames]
ax.bar(cnames, means, color=colors_ds, alpha=0.8, width=0.5)
ax.set_ylabel('Mean assertions deleted per UNKNOWN event')
ax.set_title('Deletion scope per UNKNOWN event\n(A: mean=6.7, max single event=20!)')
for i, m in enumerate(means):
    ax.text(i, m + 0.1, f'{m:.1f}', ha='center', fontweight='bold')
ax.set_ylim(0, 9)
ax.axhline(1, color='green', linestyle='--', alpha=0.5, label='Targeted (scope=1)')
ax.legend(fontsize=8)

ax = axes[2]
panic = [deletion_scope_data[c]['panic_pct'] for c in cnames]
targeted = [deletion_scope_data[c]['targeted_pct'] for c in cnames]
x = np.arange(len(cnames)); w = 0.35
b1 = ax.bar(x - w/2, panic, width=w, label='Panic (scope≥3)', color='#e74c3c', alpha=0.8)
b2 = ax.bar(x + w/2, targeted, width=w, label='Targeted (scope=1)', color='#2ecc71', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(cnames)
ax.set_ylabel('%')
ax.set_title('Panic vs Targeted deletion\n(K: 80% targeted — NL contract acts as a reference anchor)')
ax.legend(); ax.set_ylim(0, 95)
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9)

fig.suptitle('Deletion Scope: how does the LLM delete assertions on UNKNOWN?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('  A/I/J: 72-76% panic deletion — CBMC times out, LLM cannot diagnose which assertion')
print('         causes UNKNOWN, so it clears all assertions in a single sweep')
print('  K:     80% targeted — the written NL contract gives the LLM a reference to pinpoint')
print('         which specific assertion conflicts with the CBMC state space constraint')
print('  M:     0 deletion events — UNKNOWN eliminated; no deletion pressure at all')

---
## Part 10: Statistical Significance

Wilcoxon signed-rank tests on per-function strict recall, paired on shared functions with GT assertions.

In [ ]:
from scipy.stats import wilcoxon

def load_recall(path):
    with open(path) as f:
        entries = json.load(f)
    return {e['func']: (e['harness_recall'] if e['harness_recall'] is not None else 0.0)
            for e in entries if e['gt_harness_count'] > 0}

cond_paths = {
    'A':      os.path.join(EVAL, 'cross_verify_results_condA_gptoss120b.json'),
    'H':      os.path.join(EVAL, 'cross_verify_results_condH_gptoss120b.json'),
    'G':      os.path.join(EVAL, 'cross_verify_results_condG_gptoss120b.json'),
    'K':      os.path.join(EVAL, 'cross_verify_results_condK_gptoss120b.json'),
    'Oracle': os.path.join(EVAL, 'cross_verify_results_condOracle_gptoss120b.json'),
    'M':      os.path.join(EVAL, 'cross_verify_results_condM_gptoss120b.json'),
}
recalls = {k: load_recall(v) for k, v in cond_paths.items()}

tests = [
    ('M', 'Oracle', 'Laziness effect reversal: M > Oracle'),
    ('M', 'K',      'PASS-recall reversal: M > K'),
    ('M', 'H',      'Bounding hint effect: M > H'),
    ('A', 'Oracle', 'Laziness effect: A > Oracle (GT assumes actually hurt recall)'),
    ('A', 'K',      'Spec-first does not improve recall: A > K'),
    ('A', 'H',      'Strategy guidance matters: A > H (same PASS rate, higher recall)'),
    ('M', 'A',      'M vs A (marginal, ns)'),
]

print(f'{"Comparison":<50} {"n":>4} {"Δmean recall":>13} {"p (one-sided)":>15} {"Sig":>6}')
print('-'*93)

results_plot = []
for c1, c2, label in tests:
    shared = sorted(set(recalls[c1]) & set(recalls[c2]))
    v1 = np.array([recalls[c1][f] for f in shared])
    v2 = np.array([recalls[c2][f] for f in shared])
    mean_diff = np.mean(v1) - np.mean(v2)
    nz = np.sum(v1 != v2)
    if nz >= 6:
        _, p = wilcoxon(v1, v2, alternative='greater')
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        results_plot.append((label[:40], mean_diff, p, sig))
        print(f'{label:<50} {len(shared):>4} {mean_diff:>+13.4f} {p:>15.4f} {sig:>6}')

print()
print('* p<0.05  ** p<0.01  *** p<0.001  (Wilcoxon signed-rank, one-sided, strict recall)')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

labels_p = [r[0] for r in results_plot]
diffs = [r[1] * 100 for r in results_plot]
pvals = [r[2] for r in results_plot]
sigs  = [r[3] for r in results_plot]

bar_colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in diffs]
ax.barh(range(len(labels_p)), diffs, color=bar_colors, alpha=0.8, height=0.6)
ax.set_yticks(range(len(labels_p)))
ax.set_yticklabels(labels_p, fontsize=9.5)
ax.set_xlabel('Δ Recall (percentage points)', fontsize=11)
ax.set_title('Wilcoxon signed-rank test results for recall differences',
             fontsize=13, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)

for i, (d, p, s) in enumerate(zip(diffs, pvals, sigs)):
    x_pos = d + (0.1 if d >= 0 else -0.1)
    ha = 'left' if d >= 0 else 'right'
    color = '#2ecc71' if s != 'ns' else '#888'
    ax.text(x_pos, i, f'p={p:.4f} {s}', va='center', ha=ha,
            fontsize=9, fontweight='bold', color=color)

ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## Part 11: Vacuity Check — Are Our Results Genuine?

**Concern**: K and Oracle have very tight `__CPROVER_assume` constraints. Could their high PASS rate be *vacuous success* — the assumes filter out all inputs, so CBMC never reaches the function call, and all assertions trivially hold on an empty path?

**Validation**: inject `__CPROVER_assert(false)` immediately before the function call. If CBMC still returns UNSAT, the call is unreachable → vacuous. If it returns SAT, the path is genuine.

In [ ]:
vacuity_summary = {}
for cond in ['K', 'Oracle', 'M']:
    path = os.path.join(EVAL, f'vacuity_check_{cond}_gptoss120b.json')
    with open(path) as f:
        entries = json.load(f)
    total = len(entries)
    vacuous  = [e['func'] for e in entries if e['vacuous'] is True]
    reachable = sum(1 for e in entries if e['vacuous'] is False)
    unknown   = sum(1 for e in entries if e['vacuous'] is None)
    vacuity_summary[cond] = {'total': total, 'vacuous': vacuous,
                              'reachable': reachable, 'unknown': unknown}

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, (cond, data) in zip(axes, vacuity_summary.items()):
    total = data['total']
    sizes = [len(data['vacuous']), data['reachable'], data['unknown']]
    labels_v = [
        f"Vacuous ({len(data['vacuous'])}/{total}, {100*len(data['vacuous'])/total:.1f}%)",
        f"Reachable/genuine ({data['reachable']}/{total})",
        f"Unknown/timeout ({data['unknown']}/{total})"
    ]
    colors_v = ['#e74c3c', '#2ecc71', '#bdc3c7']
    ax.pie(sizes, labels=labels_v, colors=colors_v, startangle=90,
           textprops={'fontsize': 8.5})
    tc = '#2ecc71' if not data['vacuous'] else '#e74c3c'
    ax.set_title(f'Condition {cond}  (n={total} SUCCESS harnesses)',
                 fontsize=11, fontweight='bold', color=tc)
    if data['vacuous']:
        ax.text(0, -1.35, f"Vacuous: {data['vacuous'][0]}",
                ha='center', fontsize=8, color='#e74c3c', style='italic')

fig.suptitle(
    'Vacuity Check Results\n'
    'K=1.5% vacuous, Oracle=1.4% vacuous, M=0% vacuous\n'
    '→ PASS-recall reversal is genuine, not a vacuous success artifact',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## Part 12: The Narrative Arc

How the findings build on each other:

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')

story = [
    ('①', 'Core Phenomenon',
     'Condition A: 92.3% of assertion deletions are triggered by CBMC UNKNOWN,\n'
     'not by FAIL (which would mean the assertion was wrong).\n'
     '→ The LLM is optimizing for verifier satisfaction, not specification correctness.',
     '#3498db'),
    ('②', 'Emergent Behaviour\n(Condition H)',
     'H gives the LLM no permission to delete assertions, yet sacrifice rate is 86.3%\n'
     'and recall is 5.6pp lower than A (p=0.026*).\n'
     '→ Sacrifice is an emergent LLM strategy under CBMC pressure — not instructed by the prompt.',
     '#e67e22'),
    ('③', 'Deliberateness\n(Condition I)',
     'Telling the LLM which assertion category is being violated raises sacrifice to 92.7%.\n'
     'Deletion becomes more targeted (panic→targeted shift).\n'
     '→ The LLM knows what it is deleting and does it anyway — goal-directed conformance behaviour.',
     '#9b59b6'),
    ('④', 'PASS–Recall Reversal\n(core empirical finding)',
     'Oracle (highest PASS 84.3%) has lowest recall (0.251). M (3rd PASS 75.3%) has highest recall (0.389).\n'
     'Ranking by PASS rate is exactly inverted by recall. Vacuity check confirms this is not an artifact.\n'
     '→ PASS rate is a misleading quality metric for LLM-generated harnesses.',
     '#c0392b'),
    ('⑤', 'Quantity Gap vs Regression Gap',
     '97% of GT misses = never generated (knowledge gap, not addressable by feedback loop changes).\n'
     'H vs A recall gap (5.6pp, p=0.026*) = regression gap (addressable by prompt design).\n'
     '→ Two independent problems requiring different solutions.',
     '#2c3e50'),
    ('⑥', 'The Only Positive Result\n(Condition M)',
     'Adding one CBMC tool-knowledge instruction ("bound scalar variables") eliminates all UNKNOWN events.\n'
     'Sacrifice drops to 0, recall rises to 0.389 — highest across all conditions.\n'
     '→ A minimal knowledge fix simultaneously improves verifier pass rate and specification quality.',
     '#2ecc71'),
]

for i, (num, title, detail, color) in enumerate(story):
    y = 0.88 - i * 0.145
    ax.add_patch(mpatches.FancyBboxPatch((0.01, y - 0.065), 0.97, 0.12,
        boxstyle='round,pad=0.01', linewidth=1.5,
        edgecolor=color, facecolor=color + '15', transform=ax.transAxes))
    ax.text(0.03, y, num, fontsize=16, fontweight='bold', color=color,
            transform=ax.transAxes, va='center')
    ax.text(0.08, y + 0.025, title, fontsize=10.5, fontweight='bold', color=color,
            transform=ax.transAxes, va='center')
    ax.text(0.08, y - 0.025, detail, fontsize=8.8, color='#2c3e50',
            transform=ax.transAxes, va='center', linespacing=1.5)

ax.set_title('Finding Narrative: from phenomenon to mechanism to intervention',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## Part 13: What Comes Next — RQ2 (Mutation Oracle)

RQ1 answers **"what does the LLM miss, and why?"**  
RQ2 answers **"do those missed assertions actually let bugs escape?"**

Method: **Mutation Oracle**
- Inject 2,584 synthetic bugs (mutants) into the 83 aws-c-common functions using universalmutator
- Run both GT harness and LLM harness against each mutant under ESBMC
- Count **silenced mutants**: bugs that GT catches but LLM misses → `GT=SAT, LLM=UNSAT`

This count is the paper's headline safety claim — grounded in a formal oracle rather than human annotation.

In [ ]:
# Inspect available mutants
total_mutants = 0
func_mutant_counts = {}
for func in os.listdir(RESULTS + '/feedback_loop_A_gptoss120b'):
    func_path = os.path.join(RESULTS, 'feedback_loop_A_gptoss120b', func)
    if not os.path.isdir(func_path): continue
    mutants = [f for f in os.listdir(func_path) if f.endswith('.mutant.c')]
    if mutants:
        func_mutant_counts[func] = len(mutants)
        total_mutants += len(mutants)

print(f'Mutants generated: {total_mutants:,} across {len(func_mutant_counts)} functions')
print()
print('Functions with most mutants:')
for func, n in sorted(func_mutant_counts.items(), key=lambda x: -x[1])[:8]:
    bar = '█' * (n // 5)
    print(f'  {func:<40} {n:>4}  {bar}')

print()
print('Next steps:')
print('  1. ESBMC parity check: verify GT harnesses give UNSAT under ESBMC')
print('  2. Run ESBMC over 2,584 mutants × 83 functions')
print('  3. silenced mutant count = GT-SAT ∩ LLM-UNSAT = paper\'s core safety result')

---
## Summary

| Experiment step | What it measures | Key result |
|----------------|-----------------|------------|
| CBMC feedback loop | Can LLMs generate passing harnesses iteratively? | Yes — but they delete correct assertions in the process |
| Sacrifice analysis | What triggers assertion deletions? | 92.3% triggered by UNKNOWN, not by incorrect assertions |
| Condition ablations (I/J) | Is sacrifice deliberate or accidental? | Deliberate — LLM knows what it deletes and still deletes it |
| PASS vs Recall | Does PASS rate reflect specification quality? | No — the two metrics are negatively correlated across conditions |
| Vacuity check | Is K/Oracle high PASS due to vacuous success? | No — only 1-2% vacuous; reversal is genuine |
| Condition M | Can fixing a knowledge gap eliminate sacrifice? | Yes — M is the only condition that improves both PASS and recall |
| **RQ2 (pending)** | **Do missed assertions let real bugs escape?** | **Pending — 2,584 mutants ready for ESBMC** |

**Central claim**: LLMs optimise for verifier satisfaction rather than specification completeness.  
Verification passing ≠ specification correct. This bias is systematic, intentional, and quantifiable.